# HydroGym, Phase 0: random search vs scipy on the calibration environment

`aquascope.gym` wraps GR4J calibration on one basin as a gym-style episode: the agent proposes a parameter vector (X1..X4), the environment runs the model over the basin's record and returns the objective (NSE here) on the calibration period as the reward, with the validation metrics in `info`. This notebook runs entirely offline on a synthetic basin whose true parameters are known, then shows how to point the same environment at a real gauged basin from the Archive.

```
pip install "aquascope[gym]"      # gymnasium is optional; the env works without it
```


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from aquascope import gym as hg

basin = hg.synthetic_basin(0, years=12)          # GR4J truth + lognormal noise, no network
basin.summary(), basin.meta["true_params"]

In [ ]:
env = hg.CalibrationEnv(basin, objective="nse", max_steps=40)
obs, info = env.reset(seed=0)
print("observation:", dict(zip(info["obs_names"], obs.round(3))))
print("action bounds:", info["param_bounds"])

# one step: the true parameters (the ceiling, since the observed flow carries noise)
obs, reward, terminated, truncated, info = env.step(basin.meta["true_params"])
print("truth NSE (calibration):", round(reward, 3), "validation:", {k: round(v, 3) for k, v in info["validation"].items() if k != "n"})

## Three baselines, same step budget

`random_search` and `nelder_mead` only see the environment (one simulation per step). `differential_evolution` gets the simulator for free (aquascope's `calibrate`, scipy DE) and submits each generation's best member as a step, so it uses many more simulator calls per step; the leaderboard reports both counts.

In [ ]:
results = {}
for name in ("random_search", "nelder_mead", "differential_evolution"):
    env.reset(seed=0)
    res = hg.BASELINES[name](env, {"seed": 0} if name != "nelder_mead" else {})
    results[name] = (res, hg.episode_table(env))
    print(f"{name:<24} best NSE {res['best_reward']:.3f}   validation NSE {res['validation']['nse']:.3f}   "
          f"steps {res['steps']}   simulator calls {res.get('simulator_calls', res['steps'])}   {res['seconds']} s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, (res, tab) in results.items():
    ax.plot(tab["step"], tab["reward"].cummax(), label=name)
ax.axhline(env.evaluate(basin.meta["true_params"])["calibration"]["nse"], color="k", ls=":", label="truth")
ax.set(xlabel="step", ylabel="best NSE so far (calibration)", ylim=(0, 1), title=basin.name)
ax.legend()
plt.show()

## Leaderboard over several basins and seeds

`run_leaderboard` plays every agent on every basin, one fresh episode per (agent, basin, seed), and returns a table you can save next to your own agent's rows.

In [ ]:
basins = [hg.synthetic_basin(s, years=10) for s in range(3)]
board = hg.run_leaderboard(basins, max_steps=25, seeds=(0, 1))
board.groupby("agent")[["best_reward", "val_nse", "val_kge", "simulator_calls", "seconds"]].mean().round(3)

## The same environment on a real basin

Any gauged station in the Archive with a catchment area is a task: `load_basin` takes the daily discharge from the discharge bundle (m3/s to mm/d over the agency's area, else BasinATLAS), fetches Open-Meteo precipitation and FAO-56 ET0 at the gauge (the Caravan exporter's forcing) and caches the frame locally. `suggest_basins` lists candidates with long records, perennial flow and little snow (GR4J here has no snow routine; snowy basins are a good way to see an agent fail honestly).

Needs the network; skip if offline.

In [ ]:
# for row in hg.suggest_basins(6):
#     print(row["source"], row["station_id"], row["area_km2"], "km2", row["n_years"], "yr")
# real = hg.load_basin("uk_ea", "013054a3-670e-49ee-afda-e0865a449197")   # Dorchester, 206 km2, 40 years
# env = hg.CalibrationEnv(real, objective="kge", max_steps=20)
# env.reset(seed=0)
# print(hg.differential_evolution(env, maxiter=8, popsize=8))
# print(env.render())

## Writing your own agent

An agent is any callable `agent(env, kwargs) -> dict`; loop over `env.step(params)` until `truncated`, and return at least `agent`, `steps`, `best_reward`, `best_params`, `validation` (copy `hg.baselines._finish`). `env.basin.frame` holds the daily `precip`, `pet`, `q_obs` in mm/d for agents that want to look at the data before proposing parameters, and `env.evaluate(params)` scores a set without spending a step. Pass your agent in `run_leaderboard(basins, {"mine": my_agent, **hg.BASELINES})`.

Issue #175 tracks the next phases (task suite across regions, more models, a leaderboard doc).